In [0]:
import numpy as np
import pandas as pd

In [0]:
df_from_sql = spark.sql("SELECT * FROM dev.mohit_gangwani.ad_dma_overall;")
df = df_from_sql.toPandas()
df.head(20)

In [0]:
def gini(ad):
    x = np.sort(np.array(ad))
    n = x.size
    if n <= 1 or np.sum(x) == 0:
        return 1.0

    index = np.arange(1, n + 1)
    a = (2.0 * np.sum(index * x))/(n * np.sum(x))
    c = (n + 1) / n
    g = a - c
    norm = n / (n - 1)
    return norm * g

In [0]:
def locality_index(values):
    if len(values) <= 1 or np.sum(values) == 0:
        return 1.0
    p = np.array(values) / np.sum(values)
    p = p[p > 0]
    H = -np.sum(p * np.log(p))
    return 1 - (H / np.log(len(p)))

In [0]:
def get_gini_entropy_df(df, ad_ids):
    x_df = df[df.ad_id.isin(ad_ids)].copy()
    x_df.sort_values(by=['ad_id', 'impression_count'], inplace=True)
    x_df.reset_index(inplace=True, drop=True)
    ndf = pd.DataFrame()
    for i, ad_id in enumerate(ad_ids):
        a = x_df[x_df['ad_id'] == ad_id]
        ndf.loc[i, 'ad_id'] = ad_id
        ndf.loc[i, 'impression_count'] = a.impression_count.sum()
        ndf.loc[i, 'dma_count'] = a.fk_dma_id.nunique()
        ndf.loc[i, 'gini'] = gini(a['impression_count'])
        ndf.loc[i, 'entropy'] = locality_index(a['impression_count'])
    return ndf

In [0]:
ad_ids_for_local = [
    'AE16546-2025-43-05892', 'AE16546-2024-45-01430', 'AE16546-2025-43-07032', 'AE16546-2025-43-05083', 'AE16546-2025-43-05730', 'AE16546-2025-43-04818', 'AE16546-2025-41-03971', 'AE16546-2023-42-05587', 'AE16546-2025-09-01406', 'AE16546-2024-42-01939'
]

get_gini_entropy_df(df=df, ad_ids=ad_ids_for_local)

In [0]:
national_ads = [
    'AE16546-2025-42-01766', 'AE16546-2025-40-02214', 'AE16546-2025-40-04118', 'AE16546-2025-36-00396', 'AE16546-2025-28-06509', 'AE16546-2025-27-01762', 'AE16546-2025-40-07328', 'AE16546-2025-40-01688', 'AE17151-2025-18-00132', 'AE16546-2025-43-00577'
]

get_gini_entropy_df(df=df, ad_ids=national_ads)

In [0]:
hybrid_ads = [
    'AE16546-2025-43-02576', 'AE16546-2025-42-02508', 'AE16546-2025-43-02335', 'AE16546-2025-05-07669', 'AE16546-2025-30-04769', 'AE16546-2025-43-03760', 'AE16546-2025-32-06856', 'AE16546-2025-42-15347', 'AE16546-2025-35-04512', 'AE16546-2025-40-06231'
]

get_gini_entropy_df(df=df, ad_ids=hybrid_ads)